# LangChain: Memory


## All Memory

**MessagesPlaceholder("history")** is a slot in the prompt template where a list of past conversation messages gets inserted at runtime. It's used to inject chat history into the prompt dynamically.

In [4]:
import os
# get home and job id
home_dir =  os.getenv('HOME')
job_id = os.getenv('SLURM_JOB_ID')

# get ollama directory or default to home
ollama_dir = os.getenv('OLLAMA_BASE_DIR', home_dir)

try:    
    with open(f"{ollama_dir}/ollama/host_{job_id}.txt") as f:
        HOST = f.read().strip()
    with open(f"{ollama_dir}/ollama/port_{job_id}.txt") as f:
        PORT = f.read().strip()
    ollama_url = f"http://{HOST}:{PORT}"
except Exception as e:
    print("[⚠️] Could not read host/port, you manually set the `ollama_url` below.")

print("The port that serve ollama is: ", ollama_url)


The port that serve ollama is:  http://bcm-dgxa100-0001:38833


In [8]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

llm = ChatOllama(base_url=ollama_url,
                model="gemma3:4b",  
                temperature=0.2)
                
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])

chain = prompt | llm

In [9]:
_store = {}
def get_history(session_id: str):
    if session_id not in _store:
        _store[session_id] = ChatMessageHistory()
    return _store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

cfg = {"configurable": {"session_id": "tue-session"}}



In [10]:
print(conversation.invoke({"input": "Hi, my name is Tue."}, config=cfg).content)

Hi Tue! It’s nice to meet you. How can I help you today? 😊 

Do you have something specific you’d like to talk about, or were you just saying hello?


In [5]:
print(conversation.invoke({"input": "What is my name?"}, config=cfg).content)

Your name is Tue! 😊 I just confirmed it with you. 

Is there anything else you’d like to tell me about yourself?


In [6]:
history = get_history("tue-session")
print(history)


Human: Hi, my name is Tue.
AI: Hi Tue! It’s nice to meet you. How can I help you today? 😊 

Do you have something specific you’d like to talk about, or were you just saying hello?
Human: What is my name?
AI: Your name is Tue! 😊 I just confirmed it with you. 

Is there anything else you’d like to tell me about yourself?
